In [ ]:
!pip install -q openpyxl pandas tqdm


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving pershop_slots.xlsx to pershop_slots.xlsx


In [ ]:
import pandas as pd
import re

# اگر اسم فایل چیز دیگری است، اینجا اصلاح کن
slots_df_raw = pd.read_excel("pershop_slots.xlsx")
slots_df_raw = slots_df_raw.dropna(subset=["annotated_text"]).reset_index(drop=True)
slots_df_raw["annotated_text"] = slots_df_raw["annotated_text"].astype(str)

print("Raw slots df:", slots_df_raw.shape)
print(slots_df_raw.head())


Raw slots df: (11065, 1)
                                      annotated_text
0                                          سلام خوبی
1                                              ممنون
2                  من یک عدد [شیر](product) می خواهم
3  [شیر](product) مخصوص [خردسالان](cat) [ماجان](b...
4                                         نه نمیخوام


In [ ]:
slot_pattern = re.compile(r'\[([^\]]+)\]\(([^)]+)\)')


In [ ]:
def strip_annotations(annotated_text: str) -> str:
    """
    [شیر](product) [کم چرب](fat) -> شیر کم چرب
    """
    return slot_pattern.sub(lambda m: m.group(1), annotated_text).strip()

def extract_gold_slots(annotated_text: str):
    """
    برمی‌گرداند لیستی از dict:
    [{"slot": "product", "value": "شیر"}, {"slot": "fat", "value": "کم چرب"}, ...]
    """
    gold = []
    for m in slot_pattern.finditer(annotated_text):
        span_text = m.group(1).strip()
        slot_name = m.group(2).strip()
        if span_text and slot_name:
            gold.append({"slot": slot_name, "value": span_text})
    return gold

records = []
for _, row in slots_df_raw.iterrows():
    ann = row["annotated_text"]
    text_raw = strip_annotations(ann)
    gold_slots = extract_gold_slots(ann)
    if not text_raw:
        continue
    records.append({
        "text": text_raw,
        "gold_slots": gold_slots
    })

slots_df = pd.DataFrame(records)
print("Parsed slots df:", slots_df.shape)
print(slots_df.head())


Parsed slots df: (11065, 2)
                                                text  \
0                                          سلام خوبی   
1                                              ممنون   
2                             من یک عدد شیر می خواهم   
3  شیر مخصوص خردسالان ماجان 200 میلی لیتری فرادما...   
4                                         نه نمیخوام   

                                          gold_slots  
0                                                 []  
1                                                 []  
2              [{'slot': 'product', 'value': 'شیر'}]  
3  [{'slot': 'product', 'value': 'شیر'}, {'slot':...  
4                                                 []  


In [ ]:
from sklearn.model_selection import train_test_split

# اگر slots_df خیلی بزرگه، یک زیرمجموعه‌ی معقول انتخاب کن
N_SAMPLES = 200  # اگر می‌خوای همه رو تست کنی، این خط رو کامنت کن و از slots_df کامل استفاده کن

if N_SAMPLES is not None and len(slots_df) > N_SAMPLES:
    slots_df_sample = slots_df.sample(n=N_SAMPLES, random_state=42).reset_index(drop=True)
else:
    slots_df_sample = slots_df.copy()

print("Number of utterances for LLM eval:", len(slots_df_sample))


Number of utterances for LLM eval: 200


In [ ]:
SLOT_LABELS = [
    "product",
    "sub-domain",
    "category",
    "type",
    "fat",
    "fresh",
    "size",
    "brand",
    "sub-brand",
    "volume",
    "weight",
    "flavor",
    "carbonate",
    "herb-content",
    "package",
    "brew-method",
    "unit-count",
    "numeric-size",
    "color",
    "color-code",
    "country",
    "percent",
    "height",
    "width",
    "length",
    "diameter",
    "material",
    "skin-type",
    "hair-type",
    "gender",
    "energy-rank",
    "capacity",
    "price",
]


In [ ]:
SLOT_LIST_STR = ", ".join(SLOT_LABELS)

def build_batch_prompt(utterances):
    items_str = ""
    for i, utt in enumerate(utterances, start=1):
        items_str += f'{i}. "{utt}"\n'

    examples = """
Here are examples:

Example 1:
User: "یه شیر کم چرب کاله یک لیتری می‌خوام"
Output:
[
  {
    "id": 1,
    "slots": [
      {"slot": "product", "value": "شیر"},
      {"slot": "fat", "value": "کم چرب"},
      {"slot": "brand", "value": "کاله"},
      {"slot": "volume", "value": "یک لیتری"}
    ]
  }
]

Example 2:
User: "برام یه برنج طارم ایرانی می‌خوای بفرستی؟"
Output:
[
  {
    "id": 1,
    "slots": [
      {"slot": "product", "value": "برنج"},
      {"slot": "category", "value": "طارم"},
      {"slot": "country", "value": "ایرانی"}
    ]
  }
]

Example 3:
User: "شامپو برای موهای خشک می‌خوام"
Output:
[
  {
    "id": 1,
    "slots": [
      {"slot": "product", "value": "شامپو"},
      {"slot": "hair-type", "value": "موهای خشک"}
    ]
  }
]

Example 4:
User: "یه آب معدنی خانواده می‌خوام"
Output:
[
  {
    "id": 1,
    "slots": [
      {"slot": "product", "value": "آب معدنی"},
      {"slot": "size", "value": "خانواده"}
    ]
  }
]

Example 5:
User: "هیچی نمی‌خوام، فقط می‌پرسم"
Output:
[
  {
    "id": 1,
    "slots": []
  }
]
"""

    return f"""
You are an expert NLU system for Persian shopping dialogues.

Your task: For each user utterance in Persian, extract all product-related slot-value pairs.

The allowed slot labels are ONLY:
{SLOT_LIST_STR}

Output format:
- Return a JSON array.
- Each element in the array must be an object with:
  - "id": the index of the utterance (1, 2, 3, ...)
  - "slots": a list of objects, each with:
      - "slot": one of [{SLOT_LIST_STR}]
      - "value": the exact text span from the utterance.

Important rules:
- Use ONLY the above slot names.
- If no slot is mentioned, use an empty list [] for "slots".
- Do NOT explain.
- Do NOT translate.
- Do NOT output anything except the JSON array.

{examples}

Now extract slots for the following utterances:

{items_str}

Return ONLY the JSON array.
"""


In [ ]:
from google.colab import userdata  # store the key in Colab Secrets as OPENAI_API_KEY
from openai import OpenAI

# اینجا API Key خودت رو قرار بده
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

def call_llm_gpt(prompt: str, model_name: str = "gpt-5.1-2025-11-13"):
    # اگر GPT-5.1 واقعی در دسترس‌ات هست، اسم مدلش رو اینجا بذار
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are a precise information extraction model."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    return response.choices[0].message.content


In [ ]:
import json
import re
import unicodedata

def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    s = unicodedata.normalize("NFKC", s)
    return s

def extract_json_array(text: str):
    m = re.search(r'\[.*\]', text, re.DOTALL)
    if not m:
        return None
    candidate = m.group(0)
    try:
        return json.loads(candidate)
    except Exception:
        return None


In [ ]:
from tqdm import tqdm

utterances = slots_df_sample["text"].tolist()
gold_slots_all = slots_df_sample["gold_slots"].tolist()

BATCH_SIZE = 10  # تعداد جملات در هر درخواست به GPT

pred_slots_all = []  # لیست پیش‌بینی‌ها برای هر utterance

for i in tqdm(range(0, len(utterances), BATCH_SIZE)):
    batch_utts = utterances[i:i+BATCH_SIZE]
    prompt = build_batch_prompt(batch_utts)
    raw_output = call_llm_gpt(prompt, model_name="gpt-5.1-2025-11-13")  # یا GPT-5.1

    js = extract_json_array(raw_output)
    if not js:
        # اگر نتونستیم JSON رو بخونیم، برای این batch اسلات‌ها رو خالی می‌ذاریم
        for _ in batch_utts:
            pred_slots_all.append([])
        continue

    # map از id → slots
    pred_map = {}
    for item in js:
        try:
            idx = int(item.get("id", -1))
            slots_list = item.get("slots", [])
            clean_slots = []
            for sl in slots_list:
                slot_name = normalize_text(sl.get("slot", ""))
                value = normalize_text(sl.get("value", ""))
                if slot_name in SLOT_LABELS and value:
                    clean_slots.append({"slot": slot_name, "value": value})
            pred_map[idx] = clean_slots
        except Exception:
            continue

    # طبق ترتیب batch
    for j in range(1, len(batch_utts) + 1):
        pred_slots_all.append(pred_map.get(j, []))

len(pred_slots_all), len(utterances)


100%|██████████| 20/20 [01:23<00:00,  4.18s/it]


(200, 200)

In [ ]:
from sklearn.metrics import f1_score

def evaluate_slot_labels_only(preds, golds):
    y_true = []
    y_pred = []

    for pred_slots_list, gold_slots_list in zip(preds, golds):
        # gold_slots_list is a list of dicts: [{'slot': 'brand', 'value': 'کاله'}, {'slot': 'fat', 'value': 'کم چرب'}, ...]
        gold_labels = set(g["slot"] for g in gold_slots_list)  # Extract slot names
        pred_labels = set(p["slot"] for p in pred_slots_list)  # Extract slot names

        for slot in gold_labels:
            y_true.append(1)
            y_pred.append(1 if slot in pred_labels else 0)

        for slot in pred_labels - gold_labels:
            y_true.append(0)
            y_pred.append(1)

    f1 = f1_score(y_true, y_pred, zero_division=0)
    return f1

In [ ]:
slot_f1 = evaluate_slot_labels_only(pred_slots_all, gold_slots_all)
print(f"Slot Label F1 Score (Presence Only): {slot_f1:.4f} ({slot_f1*100:.2f} %)")

Slot Label F1 Score (Presence Only): 0.7260 (72.60 %)
